# E0/E1 통제된 이상 주입 평가 — 통합 Isolation Forest

**목적:** 동일 Test 데이터의 일부 연속 구간에 인위적 출력 저하를 주입하고 E0(인버터 4특징), E1(+ASOS 3특징)의 탐지 결과를 비교한다.

- 학습 완료 모델을 **재학습하지 않음**. Validation에서 설정한 임계값을 고정한다.
- 실제 고장 레이블은 없으므로 지표는 **합성 이상 주입 탐지율**이다. 원본 Test의 미주입 구간 경보율은 진정한 FPR이 아니다.
- 출력이 감소하면 `energy_ratio`도 같이 감소시키고, `power_ratio_diff`와 `ratio_ma_60min`을 시각 기반으로 일관되게 수정한다.
- **주의:** 주입한 데이터의 누적 계량값(`accu_energy`)은 수정하지 않는다. 합성 에너지 증분은 별도 컬럼으로만 출력한다. 합성 CSV는 물리 계측 원본이 아니며 운영 데이터로 재사용하지 않는다.
- 동일 테스트 행에 각 시나리오를 **독립적으로** 주입하며, 시나리오 간 이상 구간은 누적되지 않는다.


In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest

ROOT = Path.cwd()  # notebook을 /mnt/data에 놓고 실행하는 경우 아래 경로로 탐색
SEARCH_ROOTS = [ROOT, ROOT / 'data', Path('/mnt/data')]
def locate(relative):
    for root in SEARCH_ROOTS:
        candidate = root / relative
        if candidate.is_file(): return candidate
    raise FileNotFoundError(f'{relative} 파일을 찾지 못했습니다. SEARCH_ROOTS를 조정하세요: {SEARCH_ROOTS}')
TEST_FILE = locate('clean_output/06_common_test.csv')
E0_MODEL_FILE = locate('e0_output/e0_unified_iforest.joblib')
E1_MODEL_FILE = locate('e1_output/e1_weather_iforest.joblib')
E0_META_FILE = locate('e0_output/e0_metadata.json')
E1_META_FILE = locate('e1_output/e1_metadata.json')
OUT = ROOT / 'e0_e1_injection_output'
OUT.mkdir(exist_ok=True, parents=True)
SEED = 42
print('Python',sys.version.split()[0], '| Test', TEST_FILE, '| Output', OUT)


In [ ]:
df = pd.read_csv(TEST_FILE, parse_dates=['timestamp']).sort_values(['source_plant','timestamp','source_row']).reset_index(drop=True)
m0, m1 = joblib.load(E0_MODEL_FILE), joblib.load(E1_MODEL_FILE)
meta0, meta1 = json.loads(E0_META_FILE.read_text(encoding='utf-8')), json.loads(E1_META_FILE.read_text(encoding='utf-8'))
f0, f1 = meta0['features_ordered'], meta1['features_ordered']
th0 = float(meta0['threshold']); th1 = float(meta1['e1_threshold'])
assert f1[:len(f0)] == f0, 'E0와 E1 공통 특징 순서가 다릅니다.'
assert all(k in df.columns for k in f1)
assert df[['source_plant','source_row']].duplicated().sum()==0, '레코드 식별키 중복'
assert df[f1].notna().all().all(), '필수 특징 결측 존재'
assert len(f0)==4 and len(f1)==7, '특징 수를 다시 검증하세요.'
print('Plant sizes:', df.groupby('source_plant').size().to_dict())
print('E0 features:', f0, '\nE1 extra features:', f1[len(f0):])
print('Thresholds:',th0,th1)


## 1. 주입 시나리오

- **step_drop**: 충분히 발전 중인 3개 연속 샘플에서 출력을 50% 감소 (단발성 급락)
- **sustained_drop**: 12개 연속 샘플에서 출력을 30% 감소 (약 1시간 지속)
- **gradual_drop**: 12개 샘플 동안 출력 저하율을 5%에서 40%까지 점진적으로 증가

주입 후보는 각 발전소별 **2~8분 간격으로 연속되는 발전 중 레코드**이며, 시나리오마다 미리 정의된 수의 에피소드를 난수 시드 고정으로 선정한다. 기존 원본 이상 경보를 제거하지 않아 상대적인 주입 전후 변화를 함께 평가한다.

In [ ]:
SCENARIOS = {
    'step_drop': {'length':3, 'start_factor':0.50, 'end_factor':0.50},
    'sustained_drop': {'length':12, 'start_factor':0.70, 'end_factor':0.70},
    'gradual_drop': {'length':12, 'start_factor':0.95, 'end_factor':0.60},
}
EPISODES_PER_PLANT = 25
MIN_RATIO = 0.20
MIN_SEPARATION = 24  # 각 에피소드 사이 최소 관측 행 간격, 약 2시간
EXPECTED_MINUTES = (2,8)

def select_episodes(base, length, count, seed):
    rng = np.random.default_rng(seed)
    chosen = []
    for plant_idx, (plant, g) in enumerate(base.groupby('source_plant',sort=True)):
        idx = g.index.to_numpy()
        gaps = g['timestamp'].diff().dt.total_seconds().div(60).to_numpy()
        active = (g['rate_output'].to_numpy(dtype=float) >= MIN_RATIO)
        # 주입 시작 이전과 주입 종료 이후의 한 행도 유효한 간격인지 검사
        viable = []
        for pos in range(1, len(idx)-length):
            if not active[pos:pos+length].all(): continue
            if not np.isfinite(gaps[pos:pos+length+1]).all(): continue
            if not ((gaps[pos:pos+length+1]>=EXPECTED_MINUTES[0]) & (gaps[pos:pos+length+1]<=EXPECTED_MINUTES[1])).all(): continue
            viable.append(pos)
        order = rng.permutation(viable)
        starts=[]
        for pos in order:
            if all(abs(pos-prev) > length+MIN_SEPARATION for prev in starts):
                starts.append(int(pos))
            if len(starts)>=count: break
        if len(starts)<count:
            raise ValueError(f'{plant}: 사용 가능한 에피소드 {len(starts)} < 요청 {count}. EPISODES_PER_PLANT를 낮추세요.')
        for k, pos in enumerate(sorted(starts),1):
            chosen.append(dict(plant=plant, episode_id=f'{plant}_{k:02d}', indices=idx[pos:pos+length]))
    return chosen

# 모든 시나리오에 공통으로 사용할 12개 길이 후보를 먼저 선정하여 비교 시점의 편향을 감소
COMMON_EPISODES = select_episodes(df, length=12, count=EPISODES_PER_PLANT, seed=SEED)
print('총 에피소드:',len(COMMON_EPISODES), ' / 각 시나리오에 동일 시작 시각 적용')


In [ ]:
def inject_scenario(original, name, config, episodes):
    x = original.copy()
    n = len(x)
    factors = np.ones(n, dtype=float)
    injected = np.zeros(n, dtype=bool)
    eids = np.array(['']*n, dtype=object)
    per_episode=[]
    for ep in episodes:
        selected = ep['indices'][:config['length']]
        fs = np.linspace(config['start_factor'],config['end_factor'],len(selected))
        factors[selected]=fs
        injected[selected]=True
        eids[selected]=ep['episode_id']
        per_episode.append({'scenario':name, 'source_plant':ep['plant'], 'episode_id':ep['episode_id'],
                            'start':str(x.loc[selected[0],'timestamp']), 'end':str(x.loc[selected[-1],'timestamp']),
                            'samples':len(selected), 'minimum_factor':float(fs.min())})
    x['is_injected']=injected
    x['episode_id']=eids
    x['synthetic_factor']=factors
    x['synthetic_output_w']=x['output'].to_numpy(dtype=float)*factors
    x['synthetic_delta_wh']=x['delta_accu'].to_numpy(dtype=float)*factors
    base_ratio=x['rate_output'].to_numpy(dtype=float)
    modified_ratio=base_ratio*factors
    x['rate_output']=modified_ratio
    x['energy_ratio']=x['energy_ratio'].to_numpy(dtype=float)*factors
    # 원래 연속 구간의 직전 샘플을 참조해 차분 특징 교정. 끊김 경계는 기존 값을 유지.
    x['power_ratio_diff']=x['power_ratio_diff'].astype(float)
    # 이동평균: 과거 60분 구간에 포함되는 주입 변화의 합만 기존 이동평균에 가감.
    # 원본 이동평균은 정제 시 time-based rolling으로 구성되므로 해당 60분 분모는 동일 규칙으로 계산해야 한다.
    # 원본의 전체 미보존 구간을 정확하게 복구할 수 없어, 각 원본 MA의 기반 샘플 수를 알 수 없는 경우
    # 영향 구간을 평가에서 제외하고, 주입행은 TEST 연속 구간의 time-window로 원본/변형 평균의 차이를 더한다.
    x['_delta_ratio']=modified_ratio-base_ratio
    x['history_valid']=True
    for plant, idx in x.groupby('source_plant',sort=False).groups.items():
        indices=np.asarray(list(idx),dtype=int)
        g=x.loc[indices]
        ts=g['timestamp']
        spacing=ts.diff().dt.total_seconds().div(60)
        valid=(spacing>=EXPECTED_MINUTES[0]) & (spacing<=EXPECTED_MINUTES[1])
        prev=np.roll(modified_ratio[indices],1)
        newdiff=modified_ratio[indices]-prev
        valid_np=valid.to_numpy()
        x.loc[indices[valid_np],'power_ratio_diff']=newdiff[valid_np]
        # MA는 원본 정의가 반드시 동일한지 metadata 없이 확정할 수 없어
        # 주입 구간이 포함된 60분 창에서 TEST 상의 12개 연속 샘플을 확보한 행만 평가.
        delta=pd.Series(x.loc[indices,'_delta_ratio'].to_numpy(), index=pd.DatetimeIndex(ts))
        delta_sum=delta.rolling('60min',min_periods=1,closed='both').sum().to_numpy()
        # window 표본 수는 E0 데이터 원본 생성 코드의 정의에 따라 달라질 수 있음:
        # 원본에 저장된 MA 대신 비슷한 결측 없는 구간에서만 재구성한 값을 사용.
        base=pd.Series(base_ratio[indices],index=pd.DatetimeIndex(ts))
        mod=pd.Series(modified_ratio[indices],index=pd.DatetimeIndex(ts))
        # 바로 이전 샘플의 수집 공백이 있는 구간에 대해 정확한 60분 MA 재구성 방지
        contiguous=(~valid).cumsum().to_numpy()
        base_ma=np.empty(len(g)); mod_ma=np.empty(len(g)); count=np.empty(len(g))
        for run in np.unique(contiguous):
            mask=contiguous==run
            base_rolling=base.loc[mask].rolling('60min',min_periods=1)
            mod_rolling=mod.loc[mask].rolling('60min',min_periods=1)
            base_ma[mask]=base_rolling.mean().to_numpy()
            mod_ma[mask]=mod_rolling.mean().to_numpy()
            count[mask]=base_rolling.count().to_numpy()
        # E0 원본 ratio_ma_60min과의 정의 차이로 주입과 무관한 점수를 바꾸지 않도록 변화분만 더함
        x.loc[indices,'ratio_ma_60min']=x.loc[indices,'ratio_ma_60min'].to_numpy()+(mod_ma-base_ma)
        # 60분 이내 주입이 있고 데이터 누락을 포함하는 창은 지표에서 제외할 수 있도록 표시
        influenced=pd.Series(injected[indices].astype(int),index=pd.DatetimeIndex(ts)).rolling('60min',min_periods=1).sum().to_numpy()>0
        x.loc[indices,'history_valid']=~(influenced & (count<2))
    x=x.drop(columns=['_delta_ratio'])
    return x,pd.DataFrame(per_episode)

def scores(model, features, frame, threshold):
    matrix=frame[features].to_numpy(dtype=np.float64)
    s=-model.score_samples(matrix)
    return s, s>=threshold


## 2. 평가 지표

- **주입행 탐지율 (injected recall)**: 실제로 값을 변경한 행 중 모델이 경보한 비율
- **에피소드 탐지율 (episode recall)**: 에피소드 구간에 경보가 한 번이라도 발생한 비율
- **순증 탐지율 (newly detected injected fraction)**: 주입 전에는 경보가 없었지만 주입 후 경보로 바뀐 주입행 비율
- **미주입 구간 경보율**: 기준 데이터의 실제 정상 여부가 확인되지 않아 *FPR이 아님*
- 주입 효과가 60분 이동평균에 남는 **후행 구간**도 별도 집계하여 정상 경보율을 과대평가하지 않도록 주의


In [ ]:
baseline0, basealert0=scores(m0,f0,df,th0)
baseline1, basealert1=scores(m1,f1,df,th1)
results=[]; episode_results=[]; changed_examples=[]
for scenario, conf in SCENARIOS.items():
    injected, events=inject_scenario(df,scenario,conf,COMMON_EPISODES)
    s0,a0=scores(m0,f0,injected,th0)
    s1,a1=scores(m1,f1,injected,th1)
    mask=injected['is_injected'].to_numpy()
    for model_name, scor, alerted, before_alert in [('E0',s0,a0,basealert0),('E1',s1,a1,basealert1)]:
        for plant in sorted(df['source_plant'].unique()):
            pm=df['source_plant'].to_numpy()==plant
            im=mask & pm
            clean=~mask & pm
            results.append(dict(scenario=scenario,model=model_name,source_plant=plant,
                                injected_rows=int(im.sum()),injected_recall=float(alerted[im].mean()),
                                newly_alerted_injected=float((alerted[im]&~before_alert[im]).mean()),
                                original_alert_rate_injected=float(before_alert[im].mean()),
                                unmodified_test_alert_rate=float(before_alert[pm].mean()),
                                unmodified_rows_after_alert_rate=float(alerted[clean].mean())))
        for ep in events.to_dict('records'):
            ids=(injected['episode_id'].to_numpy()==ep['episode_id'])
            episode_results.append(dict(scenario=scenario,model=model_name,source_plant=ep['source_plant'],
                        episode_id=ep['episode_id'],start=ep['start'],end=ep['end'],
                        detected=bool(alerted[ids].any()),baseline_detected=bool(before_alert[ids].any()),
                        injected_rows=int(ids.sum()),detected_rows=int(alerted[ids].sum())))
    # 전량 CSV 대신 이상 주입행만 보존 (시나리오별 150~600행)
    subset=injected.loc[mask,['source_plant','source_row','timestamp','episode_id','synthetic_factor','output',
             'synthetic_output_w','delta_accu','synthetic_delta_wh','rate_output','energy_ratio','power_ratio_diff','ratio_ma_60min']].copy()
    subset['e0_before_alert']=basealert0[mask]
    subset['e0_after_alert']=a0[mask]
    subset['e1_before_alert']=basealert1[mask]
    subset['e1_after_alert']=a1[mask]
    subset['e0_after_score']=s0[mask]
    subset['e1_after_score']=s1[mask]
    subset.insert(0,'scenario',scenario)
    changed_examples.append(subset)

summary=pd.DataFrame(results)
episode_df=pd.DataFrame(episode_results)
summary=summary.merge(episode_df.groupby(['scenario','model','source_plant'])['detected'].mean().rename('episode_recall').reset_index(),on=['scenario','model','source_plant'])
summary.to_csv(OUT/'01_synthetic_detection_summary.csv',index=False,encoding='utf-8-sig')
episode_df.to_csv(OUT/'02_episode_results.csv',index=False,encoding='utf-8-sig')
pd.concat(changed_examples,ignore_index=True).to_csv(OUT/'03_injected_samples.csv',index=False,encoding='utf-8-sig')
print(summary.round(4).to_string(index=False))


In [ ]:
# 전체 합계 (발전소에 동일 수의 에피소드가 주입됨)
agg=summary.groupby(['scenario','model'],as_index=False).agg(injected_recall=('injected_recall','mean'),episode_recall=('episode_recall','mean'),newly_alerted_injected=('newly_alerted_injected','mean'),unmodified_test_alert_rate=('unmodified_test_alert_rate','mean'))
agg.to_csv(OUT/'04_overall_comparison.csv',index=False,encoding='utf-8-sig')
display(agg.round(4))


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,5))
labels=[f"{s}\n{m}" for s,m in zip(agg['scenario'],agg['model'])]
ax.bar(range(len(agg)),agg['injected_recall'].to_numpy())
ax.set_xticks(range(len(agg)),labels,rotation=0)
ax.set_ylim(0,1);ax.set_ylabel('Injected-row recall')
ax.set_title('E0 / E1 controlled anomaly injection (synthetic only)')
fig.tight_layout()
fig.savefig(OUT/'05_injection_recall.png',dpi=150)
plt.show()


In [ ]:
manifest={'input_test':str(TEST_FILE),'model_e0':str(E0_MODEL_FILE),'model_e1':str(E1_MODEL_FILE),
          'seed':SEED,'episodes_per_plant':EPISODES_PER_PLANT,
          'scenarios':SCENARIOS,'thresholds':{'E0':th0,'E1':th1},
          'metric_scope':'synthetic anomalies only; baseline unmodified alerts are NOT false positive rates',
          'notes':'No model refitting, no validation/test leakage, same test rows and episode onset across E0/E1.'}
(OUT/'00_run_metadata.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('완료:',OUT.resolve())


## 해석 주의사항

1. 주입 시나리오는 실제 설비 고장 데이터가 아니므로 **실제 고장 탐지 정확도라고 보고하지 않습니다.**
2. E0와 E1의 판정 임계값은 별도 Validation 분포에서 정한 값이며, 두 모델의 이상 점수 절대값을 직접 비교하지 않습니다.
3. 시간별 기상 데이터를 과거 시점 보간하여 사용한 **오프라인 실험**입니다.
4. 이 노트북은 출력저하·급락·점진저하를 검증하며, 전압/전류 고장과 설비 고장 원인을 검증하지 않습니다.
5. 기존 정제 데이터의 `ratio_ma_60min`은 원본의 연속 관측 상태를 기반으로 생성되었습니다. 본 노트북은 같은 시간창의 **주입 효과에 의한 차이만 반영**하므로 초기 구간의 과거 데이터가 없는 경계에서는 엄밀한 복원이 제한될 수 있습니다.
6. `accu_energy`는 원본 값을 보존하며 출력/에너지 주입 데이터는 별도 합성 필드에서만 관리합니다.
